In [ ]:
import io

import pymupdf
import ollama
from pathlib import Path
from PIL import Image

MODEL_NAME = "qwen3-vl:4b-instruct"
PDF_PATH = Path("images/산업안전보건법 주요 개정내용(제조업).pdf")
PAGE_NUMBER = 3  # 사람이 보는 페이지 번호 (1-based)
PROMPT = "이 이미지는 PDF 문서의 한 페이지다. 페이지에 있는 텍스트 내용을 그대로 추출해서 출력해줘. 다른 설명은 하지 말고 추출한 텍스트만 출력해."

# PDF 페이지를 이미지로 렌더링 (해상도를 높이기 위해 확대 행렬 적용)
doc = pymupdf.open(PDF_PATH)
page = doc.load_page(PAGE_NUMBER - 1)  # pymupdf는 0-based 인덱스
zoom_matrix = pymupdf.Matrix(2, 2)  # 2배 확대 렌더링
pix = page.get_pixmap(matrix=zoom_matrix)

img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
buf = io.BytesIO()
img.save(buf, format="PNG")
image_bytes = buf.getvalue()

doc.close()

response = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": PROMPT,
            "images": [image_bytes],
        }
    ],
)

extracted_text = response["message"]["content"].strip()
print(extracted_text)

pdf의 텍스트 구조를 상실하게 된다. vlm의 추론 능력을 활용하여 pdf의 구조를 유지해 보자.

In [ ]:
MARKDOWN_PROMPT = (
    "이 이미지는 PDF 문서의 한 페이지다. 페이지에 있는 텍스트 내용을 마크다운 문법을 활용해서 정리해줘.\n"
    "제목은 #, 소제목은 ##/### 등으로, 목록은 -, 표는 마크다운 표 문법으로 표현해줘.\n"
    "다른 설명은 하지 말고 마크다운으로 정리된 결과만 출력해."
)

response_md = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": MARKDOWN_PROMPT,
            "images": [image_bytes],
        }
    ],
)

extracted_markdown = response_md["message"]["content"].strip()
print(extracted_markdown)
